In [0]:
import re
from pyspark.sql import functions as F

In [0]:
locations_df = spark.table("domiciliarycare.bronze.cqc_kent_locations")
providers_df = spark.table("domiciliarycare.bronze.cqc_kent_providers")

print(f"Locations: {locations_df.count()} rows, {len(locations_df.columns)} columns")
print(f"Providers: {providers_df.count()} rows, {len(providers_df.columns)} columns")

In [0]:
locations_df.display()

In [0]:
providers_df.display()

In [0]:
providers_df.display()

In [0]:
def check_postcodes(df, loc_id, postcode):
    df_with_check = df.withColumn(
        "valid",
        F.col(postcode).rlike(r"^[A-Z]{1,2}[0-9][A-Z0-9]?\s[0-9][A-Z]{2}$")
    ).withColumn(
        "issue",
        F.when(F.col(postcode).isNull() | (F.trim(F.col(postcode)) == ""), "not available")
         .when(~F.col("valid"), "mismatch")
         .otherwise(F.lit(None))
    )

    invalid_rows = df_with_check.filter(~F.col("valid")).select(loc_id, postcode, "issue")
    print(f"{invalid_rows.count()} invalid out of {df.count()} total.")
    return invalid_rows

In [0]:
check_postcodes(locations_df, loc_id="location_id", postcode="postal_code").show(truncate=False)
check_postcodes(providers_df, loc_id="provider_id", postcode="postal_code").show(truncate=False)

In [0]:
def is_effectively_null(col_name):
        return (
        F.col(col_name).isNull() |
        (F.trim(F.col(col_name)) == "") |
        (F.trim(F.col(col_name)) == "None") |
        (F.trim(F.col(col_name)) == "nan")
    )

In [0]:
null_count_loc = locations_df.select([F.count(F.when(is_effectively_null(c), c)).alias(c) for c in locations_df.columns]).display()

In [0]:
null_count_pro = providers_df.select([F.count(F.when(is_effectively_null(c), c)).alias(c) for c in providers_df.columns]).display()

In [0]:
# write code to remove the null values columns

In [0]:
good_rating_org = locations_df.filter(((F.col("overall_rating") == "Good") | (F.col("overall_rating") == "Requires improvement"))) \
    .select("name","postal_code","region","overall_rating","latitude","longitude")

good_rating_org.display()

In [0]:
good_org = locations_df.filter(F.col("overall_rating") == "Good") \
    .select("name","postal_code","region","overall_rating","latitude","longitude")
good_org.display()

In [0]:
Outstanding_org = locations_df.filter(F.col("overall_rating") == 'Outstanding') \
    .select("name","postal_code","region","overall_rating","latitude","longitude")
Outstanding_org.display()

In [0]:

improve_org = locations_df.filter(F.col("overall_rating") == 'Requires Improvement') \
    .select("name","postal_code","region","overall_rating","latitude","longitude")
improve_org.display()